# 00 · MNIST warm-up — how images work in Python

Before the harder emotion problem, we warm up on **MNIST** — the field's
"hello world": 28×28 grayscale handwritten digits, labels 0–9. It is small,
clean, and (nearly) balanced, so it isolates *technique* from *data mess*.

The muscle we build here — `load → inspect → reshape → visualize → rescale` — is
exactly what Phase 1 does for FER-2013, just on an easier dataset. Even this
warm-up reads `seed` from `config.yaml` and keeps **train/test discipline**
(we never peek at test).

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.emotion_detector.utils.config import load_config

cfg = load_config(ROOT / "config.yaml")
SEED = cfg["global"]["seed"]
rng = np.random.default_rng(SEED)
EDA_DIR = Path(ROOT) / cfg["paths"]["results_dir"] / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)
print(f"seed = {SEED}")

## 1. Load MNIST

`keras.datasets.mnist.load_data()` downloads the dataset once (cached in
`~/.keras/datasets`) and returns it **already split** into train and test — a
provided split we respect, exactly like FER-2013's `Usage` column.

In [ ]:
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
print(f"train images {x_train.shape}, labels {y_train.shape}")
print(f"test  images {x_test.shape}, labels {y_test.shape}")

## 2. Inspect shapes, dtypes, value ranges

Each image is a 2-D array of intensities — the same "image = matrix of numbers"
idea as FER-2013, just 28×28 instead of 48×48.

In [ ]:
print(f"dtype        : {x_train.dtype}")
print(f"pixel range  : [{x_train.min()}, {x_train.max()}]")
print(f"one image    : {x_train[0].shape}  ({x_train[0].size} pixels)")
print(f"labels       : {np.unique(y_train)} (dtype {y_train.dtype})")
print(f"NaNs         : {np.isnan(x_train).any()}  (uint8 can't be NaN — clean by construction)")

## 3. Label distribution — nearly balanced

Unlike FER-2013 (16× imbalance, Disgust 1.5 % vs Happy 25 %), MNIST's ten digits
are roughly even. That balance is *why* MNIST is a clean testbed — no class-weight
or resampling needed, so any accuracy change is about technique, not imbalance.

In [ ]:
digits, counts = np.unique(y_train, return_counts=True)
for d, c in zip(digits, counts):
    print(f"  digit {d}: {c:>6,} ({c / len(y_train):.1%})")
print(f"  imbalance ratio (max/min): {counts.max() / counts.min():.2f}x")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(digits, counts, color="steelblue")
ax.set_xticks(digits)
ax.set_xlabel("digit"); ax.set_ylabel("count")
ax.set_title("MNIST training label distribution (nearly balanced)")
plt.tight_layout()
fig.savefig(EDA_DIR / "mnist_label_distribution.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'mnist_label_distribution.png'}")
plt.show()

## 4. Visualize a grid of digits

Turn the `(28, 28)` arrays back into viewable images with `imshow(cmap="gray")` —
the same decode step as FER-2013. Sampling is seeded for reproducibility.

In [ ]:
idx = rng.choice(len(x_train), size=40, replace=False)
fig, axes = plt.subplots(4, 10, figsize=(12, 5))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(x_train[i], cmap="gray", vmin=0, vmax=255)
    ax.set_title(str(int(y_train[i])), fontsize=9)
    ax.axis("off")
fig.suptitle("Random MNIST training digits (labelled)", fontsize=13)
plt.tight_layout()
fig.savefig(EDA_DIR / "mnist_sample_grid.png", dpi=120, bbox_inches="tight")
print(f"Saved: {EDA_DIR / 'mnist_sample_grid.png'}")
plt.show()

## 5. Rescale to `[0, 1]` — the same normalization as #20

Dividing by 255 maps `uint8 [0, 255]` → `float [0, 1]`, exactly what
`RescalePreprocessor` (#20) does for FER-2013: it keeps inputs small and centered
so CNN gradients stay well-conditioned. The histogram *shape* is unchanged; only
the axis compresses.

In [ ]:
x_train_scaled = x_train.astype("float32") / 255.0
x_test_scaled = x_test.astype("float32") / 255.0  # same transform on test — no leakage (÷255 is stateless)

print(f"before: {x_train.dtype}, range [{x_train.min()}, {x_train.max()}]")
print(f"after : {x_train_scaled.dtype}, range [{x_train_scaled.min():.1f}, {x_train_scaled.max():.1f}]")
assert x_train_scaled.max() <= 1.0 and x_train_scaled.min() >= 0.0
print("\u2713 rescaled to [0, 1] float32")

## 6. How MNIST mirrors FER-2013

| | MNIST | FER-2013 |
|---|---|---|
| Image | 28×28 grayscale | 48×48 grayscale |
| Pixels | uint8 `[0, 255]` | uint8 `[0, 255]` |
| Classes | 10 digits | 7 emotions |
| Balance | ~even (1.24×) | skewed (16×) |
| Split | provided train/test | `Usage` column |
| Cleanliness | clean, centered | label noise, non-faces, lighting variance |

**Same muscle, harder problem.** The `load → inspect → reshape → visualize →
rescale` steps here are identical to Phase 1; FER-2013 just adds the messy parts
(imbalance, duplicates, label noise, contrast) that Phases 2–3 handle. MNIST is
the on-ramp — and later the clean baseline the PCA + logistic-regression comparison
(data.md §6) runs on.